# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

**1.** `5 (2, 3)` — массив содержит числа 0–5 в форме 2×3; второй ряд, третий столбец — 5.

**2.** `1 8` — `argmax()` возвращает позицию первого максимума (индексация с нуля), а `max()` — его значение.

**3.** `99` — NumPy-срез `a[::3]` является *представлением*, а не копией. Поэтому изменение `b[0]` меняет `a[0]`.

**4.** `2 0.5` — условию `a > 3` соответствуют два из четырёх элементов; `True` считается как 1.

**5.** `(2, 3) 0.25` — матрица нулей имеет 2 строки и 3 столбца; `linspace(0, 1, 5)` создаёт `[0, 0.25, 0.5, 0.75, 1]`.

*Проверка:* эти ответы получены чтением кода; дальше можно проверить их запуском.

In [1]:
# Загрузка файла: используйте orders.csv из LMS.
# Поддерживается также имя orders (1).csv, которое иногда создаёт браузер.
import os
import shutil

if not os.path.exists("orders.csv"):
    if os.path.exists("orders (1).csv"):
        shutil.copyfile("orders (1).csv", "orders.csv")
    else:
        try:
            from google.colab import files
            uploaded = files.upload()
            if "orders.csv" not in uploaded and len(uploaded) == 1:
                uploaded_name = next(iter(uploaded))
                if os.path.exists(uploaded_name):
                    shutil.copyfile(uploaded_name, "orders.csv")
        except ImportError:
            print("Положите orders.csv рядом с ноутбуком")

print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders (1).csv to orders (1).csv
orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [3]:
# База: используем NumPy-маски, а не цикл по заказам.

# 1. Максимальная сумма и позиция соответствующей строки.
max_index = int(amount.argmax())
print("1. Индекс самого дорогого заказа:", max_index)
print("   Сумма:", f"{amount[max_index]:,.0f} ₸")
print("   Клиент:", cust[max_index], "| город:", city[max_index])
print("   Для менеджера: максимальный заказ сделал тестовый клиент TEST, его нужно проверять отдельно.")

# 2. True выбирает оплаченные; ~paid инвертирует маску.
paid_revenue = amount[paid].sum()
unpaid_amount = amount[~paid].sum()
print("\n2. Выручка оплаченных:", f"{paid_revenue:,.0f} ₸")
print("   Сумма неоплаченных:", f"{unpaid_amount:,.0f} ₸")
print("   Для менеджера: отменённые заказы не являются полученной выручкой.")

# 3. Булева маска по городу; mean() — доля True среди всех записей.
shymkent_mask = city == "Шымкент"
shymkent_count = int(shymkent_mask.sum())
shymkent_share = shymkent_mask.mean() * 100
print("\n3. Заказов в Шымкенте:", shymkent_count)
print("   Доля от всех заказов:", f"{shymkent_share:.2f}%")
print("   Для менеджера: на Шымкент приходится около 15% заказов.")

1. Индекс самого дорогого заказа: 1001
   Сумма: 9,999,990 ₸
   Клиент: TEST | город: Алматы
   Для менеджера: максимальный заказ сделал тестовый клиент TEST, его нужно проверять отдельно.

2. Выручка оплаченных: 101,909,360 ₸
   Сумма неоплаченных: 7,974,000 ₸
   Для менеджера: отменённые заказы не являются полученной выручкой.

3. Заказов в Шымкенте: 804
   Доля от всех заказов: 14.72%
   Для менеджера: на Шымкент приходится около 15% заказов.


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [4]:
# Стандарт

# 4. 15% скидка на ОПЛАЧЕННЫЕ заказы дороже 50 000 ₸.
large_paid = paid & (amount > 50_000)
without_test = large_paid & (cust != "TEST")
loss_all = amount[large_paid].sum() * 0.15
loss_without_test = amount[without_test].sum() * 0.15
print("4. Потери от скидки по всем строкам:", f"{loss_all:,.2f} ₸")
print("   Без клиента TEST:", f"{loss_without_test:,.2f} ₸")
print("   Разница:", f"{loss_all / loss_without_test:.2f} раза")
print("   Для менеджера: три аномальных тестовых заказа искусственно завышают стоимость акции.")

# 5. Доля заказов, доставленных за 1–2 дня, в каждом городе.
# Условие (days >= 1) & (days <= 2) автоматически исключает nan.
print("\n5. Доставка за 1–2 дня по городам:")
fast_delivery_rates = {}
for place in np.unique(city):  # По условию можно проходить циклом по 6 городам.
    in_city = (city == place)
    fast = (days >= 1) & (days <= 2) & in_city
    percent = fast.sum() / in_city.sum() * 100  # Доля от ВСЕХ заказов города.
    fast_delivery_rates[place] = float(percent)
    print(f"   {place:10s} {int(fast.sum()):4d} / {int(in_city.sum()):4d} = {percent:.2f}%")
worst_city = min(fast_delivery_rates, key=fast_delivery_rates.get)
print("   Для менеджера: самая низкая доля быстрой доставки —",
      f"{worst_city} ({fast_delivery_rates[worst_city]:.2f}%); нужны улучшения логистики.")
print("   Примечание: записи без delivery_days (nan) не считаются быстрой доставкой и входят в общее число заказов.")

# 6. Ошибки в avg = amount.mean():
#    (а) учитывает отменённые заказы, (б) включает тестовые покупки TEST.
wrong_avg = amount.mean()
real_paid = paid & (cust != "TEST")
correct_avg = amount[real_paid].mean()
print("\n6. Средний чек (ошибочный, все заказы):", f"{wrong_avg:,.2f} ₸")
print("   Средний чек оплаченных заказов без TEST:", f"{correct_avg:,.2f} ₸")
print("   Для менеджера: средний чек нельзя считать по отменённым и тестовым покупкам.")
print("   Дополнение: в датасете также встречаются дубликаты, для итоговой аналитики их нужно обработать отдельно.")

4. Потери от скидки по всем строкам: 5,702,422.50 ₸
   Без клиента TEST: 1,202,427.00 ₸
   Разница: 4.74 раза
   Для менеджера: три аномальных тестовых заказа искусственно завышают стоимость акции.

5. Доставка за 1–2 дня по городам:
   Актобе      247 /  397 = 62.22%
   Алматы     1412 / 2188 = 64.53%
   Астана      883 / 1390 = 63.53%
   Атырау       94 /  146 = 64.38%
   Караганда   365 /  538 = 67.84%
   Шымкент     240 /  804 = 29.85%
   Для менеджера: самая низкая доля быстрой доставки — Шымкент (29.85%); нужны улучшения логистики.
   Примечание: записи без delivery_days (nan) не считаются быстрой доставкой и входят в общее число заказов.

6. Средний чек (ошибочный, все заказы): 20,114.11 ₸
   Средний чек оплаченных заказов без TEST: 14,672.39 ₸
   Для менеджера: средний чек нельзя считать по отменённым и тестовым покупкам.
   Дополнение: в датасете также встречаются дубликаты, для итоговой аналитики их нужно обработать отдельно.


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [5]:
# Челлендж
import time

# 7. Сравним сумму оплаченных: обычный цикл против NumPy-маски.
# Цикл по заказам здесь нужен именно как метод сравнения скорости.
repeat = 200  # Повторяем много раз, чтобы время было легче измерить.
start = time.perf_counter()
for _ in range(repeat):
    sum_loop = 0.0
    for value, is_paid in zip(amount, paid):
        if is_paid:
            sum_loop += value
loop_time = (time.perf_counter() - start) / repeat

start = time.perf_counter()
for _ in range(repeat):
    sum_numpy = amount[paid].sum()
numpy_time = (time.perf_counter() - start) / repeat
speedup = loop_time / numpy_time
assert np.isclose(sum_loop, sum_numpy)
print("7. Сумма циклом:", f"{sum_loop:,.0f} ₸")
print("   Сумма через NumPy:", f"{sum_numpy:,.0f} ₸")
print("   Цикл:", f"{loop_time * 1000:.4f} мс")
print("   NumPy:", f"{numpy_time * 1000:.4f} мс")
print("   NumPy быстрее примерно в", f"{speedup:.1f} раза")
print("   Для менеджера: векторные операции ускоряют обработку заказов; точное ускорение зависит от компьютера.")

# 8. A/B: обе группы содержат по 1000 человек.
np.random.seed(42)
a = np.random.rand(1000) < 0.05
b = np.random.rand(1000) < 0.06
print("\n8. В первом A/B-тесте: A =", int(a.sum()), "покупок, B =", int(b.sum()), "покупок.")

b_better = 0
for _ in range(100):
    a_repeat = np.random.rand(1000) < 0.05
    b_repeat = np.random.rand(1000) < 0.06
    if b_repeat.sum() > a_repeat.sum():
        b_better += 1
print("   B показала больше покупок в", b_better, "из 100 повторных тестов.")
print("   Для менеджера: группа B чаще выигрывает из-за более высокой вероятности покупки, но не всегда.")
print("   При 1000 клиентах в каждой группе случайные колебания заметны: одного теста недостаточно,")
print("   чтобы уверенно оценить небольшой прирост с 5% до 6%; нужен анализ статистической значимости и мощности.")

7. Сумма циклом: 101,909,360 ₸
   Сумма через NumPy: 101,909,360 ₸
   Цикл: 3.9160 мс
   NumPy: 0.0802 мс
   NumPy быстрее примерно в 48.8 раза
   Для менеджера: векторные операции ускоряют обработку заказов; точное ускорение зависит от компьютера.

8. В первом A/B-тесте: A = 54 покупок, B = 58 покупок.
   B показала больше покупок в 84 из 100 повторных тестов.
   Для менеджера: группа B чаще выигрывает из-за более высокой вероятности покупки, но не всегда.
   При 1000 клиентах в каждой группе случайные колебания заметны: одного теста недостаточно,
   чтобы уверенно оценить небольшой прирост с 5% до 6%; нужен анализ статистической значимости и мощности.
